# 불법 투기(쓰레기 무단 투기) 감지 - 학습 & 실시간 영상 파이프라인

이 노트북은 네 부분으로 구성됩니다.

1. **Roboflow에서 쓰레기 데이터셋 다운로드** (API 키 입력) — "봉투(정상 배출)"와 "낱개 쓰레기(위반)"를 구분할 수 있는 데이터셋
2. **YOLOv8 학습 & 테스트** — 물체가 "봉투"인지 "낱개 쓰레기"인지 라벨을 붙이는 용도
3. **MobileNet-SSD 사람 판별 모델 다운로드** — 학습이 필요 없는 사전학습 모델. 화면 속 사람 위치를 찾아서, 얼마나 오래 같은 자리에 머무르는지 추적하는 용도
4. **실시간 영상 파이프라인** (아래 "동작 방식" 참고) + **로직 테스트 & 샘플 영상 테스트**

## 동작 방식

1. 매 프레임 **이중 배경 모델**(빠르게 적응하는 단기 모델 + 천천히 적응하는 장기 모델)을 갱신해서,
   "누군가 새로 내려놓고 안 치운 물체"의 위치를 찾아둡니다.
2. **1초 간격으로** MobileNet-SSD로 화면 속 사람 위치를 찾고, 각 사람이 같은 자리에 얼마나
   머물렀는지(체류 시간)를 추적합니다. 이번 체크 타이밍에 특별히 지속되는 물체나 오래 머문
   사람이 없으면(=변동 없음) 그냥 넘어갑니다 — 아무 것도 저장하지 않습니다.
3. 어떤 사람이 **설정한 시간(기본 8초) 이상** 같은 자리에 머물렀으면, 그 사람 근처에 방금
   생긴 물체 후보가 있는지 확인합니다.
   - 물체가 없다면 아직 판단하지 않고 다음 체크 타이밍에 다시 확인합니다.
   - 물체가 있다면 학습된 YOLOv8 모델로 그 물체를 분류합니다.
4. 분류 결과에 따라:
   - **봉투(정상 배출)** 로 분류되면 → **문제 없음**. 아무 것도 저장하지 않습니다.
   - **낱개 쓰레기(무단투기)** 로 분류되면 → **위반 확정**. 화면 전체가 아니라 **그 사람 영역만
     잘라서(crop)** 사진으로 저장하고, CSV 로그에 기록합니다.

기존에 쓰시던 "배경 차분 + 정지 객체 추적" 골격은 유지하면서, ① 배경 모델을 단일 모델 →
단기/장기 이중 모델로, ② 8초 기준을 "물체 지속시간" → "사람 체류시간"으로, ③ 쓰레기 판정을
"쓰레기냐 아니냐" → "봉투냐 낱개 쓰레기냐"로, ④ 확정 시 저장물을 "사람 마스킹된 전체 프레임" →
"위반자만 크롭한 사진"으로 바꾼 구조입니다.

> ⚠️ Roboflow 데이터셋 다운로드는 `api.roboflow.com`, MobileNet-SSD 다운로드는
> `raw.githubusercontent.com` 접속이 필요합니다. 이 노트북을 실행하는 환경(Colab,
> 로컬 Jupyter 등)에 인터넷이 열려 있어야 합니다.

## 0. 설치

In [ ]:
!pip install -q ultralytics roboflow opencv-python matplotlib

## 1. Roboflow에서 쓰레기 데이터셋 다운로드

1. https://roboflow.com 에서 무료 회원가입
2. 우측 상단 프로필 -> **Settings -> API Keys** 에서 Private API Key 복사
3. 아래 `ROBOFLOW_API_KEY` 에 붙여넣기

**중요**: "봉투(정상)"와 "낱개 쓰레기(위반)"를 구분하려면 데이터셋에 `garbage_bag`처럼
봉투를 따로 구분하는 클래스가 있어야 합니다. 기본값으로는 `trash` / `garbage` / `garbage_bag`
클래스가 모두 있는 공개 데이터셋을 넣어뒀습니다. 다른 데이터셋으로 바꾸려면 워크스페이스/
프로젝트/버전을 바꾸고, 아래 6-3 섹션의 `BAG_KEYWORDS` 를 그 데이터셋의 실제 클래스 이름에
맞게 조정하세요(클래스 이름에 "bag" 또는 "봉투"라는 단어가 들어있으면 자동으로 "봉투"로 인식됩니다).

- `jadetestset1` / `final-garbage-detection-6xzdu` (v1) — trash, garbage, garbage_bag 등 (1,539장) *(기본값)*
- `prometheuscctv` / `illegal-dumping_object-detection` (v2) — garbage, person, bin (1,692장, 봉투 클래스 없음)
- `silvr` 워크스페이스의 `garbage bag`(2.89k장) 프로젝트 — 봉투만 있는 데이터셋. `garbage`류 데이터셋과 합쳐서 쓰면 좋습니다.

각 프로젝트 페이지(`https://universe.roboflow.com/<workspace>/<project>`)에서
정확한 workspace/project 이름, 클래스 목록, 최신 버전 번호를 꼭 확인하세요.

In [ ]:
import os
from roboflow import Roboflow

ROBOFLOW_API_KEY = os.environ["ROBOFLOW_API_KEY"]  # <- 반드시 본인 키로 교체
WORKSPACE = "jadetestset1"
PROJECT = "final-garbage-detection-6xzdu"
VERSION = 1
FORMAT = "yolov8"  # ultralytics(YOLOv8) 학습용 포맷

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download(FORMAT)

print("다운로드 위치:", dataset.location)
print("data.yaml:", f"{dataset.location}/data.yaml")

## 2. YOLOv8 학습 (봉투 vs 낱개 쓰레기 분류용)

- GPU가 있으면(Colab에서 런타임 유형을 GPU로 설정) 훨씬 빠릅니다.
- `epochs`, `imgsz`, `batch`은 데이터셋 크기/환경에 맞게 조절하세요.
- 이 모델은 **"사람인지 아닌지"를 판단하는 용도가 아니라**, MobileNet-SSD가 찾아낸 "오래
  머문 사람" 근처에 새로 생긴 물체가 **봉투인지 낱개 쓰레기인지**를 분류하는 용도로만
  사용됩니다.

In [ ]:
from ultralytics import YOLO

EPOCHS = 50
IMGSZ = 640
BATCH = 16

model = YOLO("yolov8n.pt")  # 가벼운 사전학습 모델에서 시작 (전이학습)
train_results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project="dumping_runs",
    name="train",
)

BEST_WEIGHTS = train_results.save_dir / "weights" / "best.pt"
print("학습된 가중치:", BEST_WEIGHTS)

## 3. 검증(mAP 등 지표 확인)

In [ ]:
trained_model = YOLO(str(BEST_WEIGHTS))
metrics = trained_model.val()
print("클래스 목록:", trained_model.names)
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

## 4. 테스트 이미지로 추론 확인

검증(val) 세트 이미지 몇 장에 대해 예측 결과를 그려서 눈으로 확인합니다.

In [ ]:
import glob
import matplotlib.pyplot as plt

val_images = glob.glob(f"{dataset.location}/valid/images/*.jpg")[:6]
if not val_images:
    val_images = glob.glob(f"{dataset.location}/valid/images/*")[:6]

results = trained_model.predict(val_images, conf=0.3, verbose=False)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, r in zip(axes.flatten(), results):
    ax.imshow(r.plot()[:, :, ::-1])  # BGR -> RGB
    ax.axis("off")
plt.tight_layout()
plt.show()

### 직접 이미지로 테스트하고 싶다면

In [ ]:
# MY_IMAGE = "/path/to/my_photo.jpg"
# result = trained_model.predict(MY_IMAGE, conf=0.3)[0]
# plt.figure(figsize=(8, 8)); plt.imshow(result.plot()[:, :, ::-1]); plt.axis("off"); plt.show()

## 5. MobileNet-SSD 사람 판별 모델 다운로드

별도 학습이 필요 없는 사전학습 모델(Caffe, VOC0712 데이터셋 기준 mAP 72.7)입니다.
OpenCV의 `cv2.dnn` 모듈만으로 바로 불러올 수 있어서 caffe를 따로 설치할 필요는 없습니다.
`person` 클래스(인덱스 15)만 사용합니다.

In [ ]:
import os
import urllib.request

import cv2

SSD_PROTOTXT_URL = "https://raw.githubusercontent.com/chuanqi305/MobileNet-SSD/master/deploy.prototxt"
SSD_WEIGHTS_URL = "https://raw.githubusercontent.com/chuanqi305/MobileNet-SSD/master/mobilenet_iter_73000.caffemodel"

if not os.path.exists("deploy.prototxt"):
    urllib.request.urlretrieve(SSD_PROTOTXT_URL, "deploy.prototxt")
if not os.path.exists("mobilenet_iter_73000.caffemodel"):
    urllib.request.urlretrieve(SSD_WEIGHTS_URL, "mobilenet_iter_73000.caffemodel")

ssd_net = cv2.dnn.readNetFromCaffe("deploy.prototxt", "mobilenet_iter_73000.caffemodel")
print("MobileNet-SSD 로드 완료")

## 6. 실시간 영상 파이프라인 구현

### 6-1. 이중 배경 모델(단기/장기) — "새로 생긴 정지 물체" 후보 검출

In [ ]:
import os
import cv2
import numpy as np


class DualBackgroundSubtractor:
    '''단기(빠르게 적응)/장기(느리게 적응) 배경 모델을 동시에 운영.

    - 장기 배경과는 다르지만(= 예전 배경에 없던 것) 단기 배경과는 같은(= 최근 몇 프레임
      동안 그대로 있어서 이미 "최근 정상"으로 흡수됨) 영역만 "정지된 새 물체" 후보로 남긴다.
    - 계속 움직이는 물체(예: 사람)는 위치가 계속 바뀌기 때문에 단기 배경도 따라잡지 못해
      fg_short 이 계속 True 로 남고, 그 결과 후보에서 자동으로 제외된다.
    '''

    def __init__(self, alpha_short=0.35, alpha_long=0.03, diff_thresh=25):
        self.bg_short = None
        self.bg_long = None
        self.alpha_short = alpha_short
        self.alpha_long = alpha_long
        self.diff_thresh = diff_thresh

    def apply(self, gray):
        gray = gray.astype("float32")
        if self.bg_short is None:
            self.bg_short = gray.copy()
            self.bg_long = gray.copy()
            return None  # 워밍업 프레임 (아직 배경 기준이 없음)

        diff_short = cv2.absdiff(gray, self.bg_short).astype("uint8")
        diff_long = cv2.absdiff(gray, self.bg_long).astype("uint8")
        _, fg_short = cv2.threshold(diff_short, self.diff_thresh, 255, cv2.THRESH_BINARY)
        _, fg_long = cv2.threshold(diff_long, self.diff_thresh, 255, cv2.THRESH_BINARY)

        self.bg_short = cv2.addWeighted(self.bg_short, 1 - self.alpha_short, gray, self.alpha_short, 0)
        self.bg_long = cv2.addWeighted(self.bg_long, 1 - self.alpha_long, gray, self.alpha_long, 0)

        # 장기 모델 기준으론 여전히 낯설지만(fg_long), 단기 모델 기준으론 이미 익숙해진(NOT fg_short) 영역
        static_mask = cv2.bitwise_and(fg_long, cv2.bitwise_not(fg_short))
        static_mask = cv2.dilate(static_mask, None, iterations=2)
        return static_mask


def boxes_from_mask(mask, min_area=800):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for c in contours:
        if cv2.contourArea(c) < min_area:
            continue
        x, y, w, h = cv2.boundingRect(c)
        boxes.append((x, y, x + w, y + h))
    return boxes


def is_on_ground(box, frame_height, ground_ratio):
    '''박스 아래쪽(y2)이 화면 하단 ground_ratio 비율 안에 있는지 (바닥 근처인지) 확인.'''
    y2 = box[3]
    return y2 >= frame_height * (1 - ground_ratio)

### 6-2. 추적기 (IoU 매칭 + 지속시간 누적)

프레임마다 새로 잡힌 박스를 이전 것들과 매칭해서 얼마나 오래 같은 자리에 있었는지
(`first_seen` ~ `last_seen`) 누적합니다. **물체 후보**와 **사람 체류시간** 양쪽에
그대로 재사용합니다(두 개의 별도 인스턴스로).

In [ ]:
class ChangeTracker:
    def __init__(self, iou_threshold=0.3, max_missed=2):
        self.iou_threshold = iou_threshold
        self.max_missed = max_missed  # 연속으로 몇 번 안 잡히면 후보에서 제거할지
        self.next_id = 0
        self.candidates = {}

    @staticmethod
    def _iou(box_a, box_b):
        ax1, ay1, ax2, ay2 = box_a
        bx1, by1, bx2, by2 = box_b
        ix1, iy1 = max(ax1, bx1), max(ay1, by1)
        ix2, iy2 = min(ax2, bx2), min(ay2, by2)
        iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
        inter = iw * ih
        area_a = (ax2 - ax1) * (ay2 - ay1)
        area_b = (bx2 - bx1) * (by2 - by1)
        union = area_a + area_b - inter
        return inter / union if union > 0 else 0.0

    def update(self, boxes, now_sec):
        matched_ids = set()
        for box in boxes:
            best_id, best_iou = None, 0.0
            for cid, cand in self.candidates.items():
                if cid in matched_ids:
                    continue
                iou = self._iou(box, cand["bbox"])
                if iou > best_iou:
                    best_id, best_iou = cid, iou
            if best_id is not None and best_iou >= self.iou_threshold:
                cand = self.candidates[best_id]
                cand["bbox"] = box
                cand["last_seen"] = now_sec
                cand["missed"] = 0
                matched_ids.add(best_id)
            else:
                cid = self.next_id
                self.next_id += 1
                self.candidates[cid] = {
                    "bbox": box, "first_seen": now_sec, "last_seen": now_sec,
                    "missed": 0, "resolved": False,
                }
                matched_ids.add(cid)

        for cid in list(self.candidates.keys()):
            if cid not in matched_ids:
                self.candidates[cid]["missed"] += 1
                if self.candidates[cid]["missed"] > self.max_missed:
                    del self.candidates[cid]

        return self.candidates

### 6-3. MobileNet-SSD 사람 탐지 + 봉투/낱개쓰레기 분류 + 사람 크롭

In [ ]:
SSD_CLASSES = ["background", "aeroplane", "bicycle", "bird", "boat",
               "bottle", "bus", "car", "cat", "chair", "cow", "diningtable",
               "dog", "horse", "motorbike", "person", "pottedplant",
               "sheep", "sofa", "train", "tvmonitor"]
SSD_PERSON_ID = SSD_CLASSES.index("person")


def make_mobilenet_ssd_detector(net, conf_threshold=0.5):
    '''cv2.dnn Net을 받아서, frame -> [사람 박스, ...] 를 반환하는 함수를 만들어준다.
    (테스트할 때는 이 함수 대신 가짜 함수로 바꿔치기하기 쉽게 분리해뒀습니다.)'''

    def _detect(frame):
        h, w = frame.shape[:2]
        blob = cv2.dnn.blobFromImage(frame, scalefactor=0.007843, size=(300, 300),
                                      mean=(127.5, 127.5, 127.5), swapRB=False)
        net.setInput(blob)
        out = net.forward()
        boxes = []
        for i in range(out.shape[2]):
            conf = float(out[0, 0, i, 2])
            cls_id = int(out[0, 0, i, 1])
            if conf < conf_threshold or cls_id != SSD_PERSON_ID:
                continue
            bx = out[0, 0, i, 3:7] * np.array([w, h, w, h])
            x1, y1, x2, y2 = bx.astype(int)
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)
            if x2 > x1 and y2 > y1:
                boxes.append((x1, y1, x2, y2))
        return boxes

    return _detect


import re

BAG_KEYWORDS = ["bag", "봉투"]  # 클래스 이름에 이 "단어"가 있으면 "정상 배출(봉투)"로 판단


def _label_matches_keyword(label, keyword):
    '''단어 단위로 매칭. "garbage" 안에 "bag"이 부분 문자열로 들어있어도("gar-BAG-e")
    오매칭되지 않도록 구분자로 토큰을 나눠서 비교합니다.'''
    tokens = re.split(r"[^0-9a-zA-Z가-힣]+", label.lower())
    return keyword.lower() in tokens


def classify_dropped_object(trash_model, roi, trash_conf, bag_keywords=BAG_KEYWORDS):
    '''ROI가 봉투인지 낱개 쓰레기인지 학습된 YOLO 모델로 분류.
    반환값: (label, score, is_bag). 아직 뭔지 판별 안 되면 (None, 0.0, None).'''
    results = trash_model.predict(roi, verbose=False, conf=trash_conf)
    r = results[0]
    if len(r.boxes) == 0:
        return None, 0.0, None
    idxs = [i for i in range(len(r.boxes.cls))
            if trash_model.names[int(r.boxes.cls[i])].lower() != "person"]
    if not idxs:
        return None, 0.0, None
    best = max(idxs, key=lambda i: float(r.boxes.conf[i]))
    label = trash_model.names[int(r.boxes.cls[best])]
    score = float(r.boxes.conf[best])
    is_bag = any(_label_matches_keyword(label, k) for k in bag_keywords)
    return label, score, is_bag


def crop_person(frame, box, pad=15):
    '''위반이 확정된 순간, 화면 전체가 아니라 그 사람 영역만 잘라낸다.'''
    x1, y1, x2, y2 = box
    h, w = frame.shape[:2]
    x1, y1 = max(0, x1 - pad), max(0, y1 - pad)
    x2, y2 = min(w, x2 + pad), min(h, y2 + pad)
    return frame[y1:y2, x1:x2].copy()


def _box_distance(box_a, box_b):
    '''두 박스 중심점 사이의 유클리드 거리 (사람과 물체가 얼마나 가까운지 판단용).'''
    ax = (box_a[0] + box_a[2]) / 2
    ay = (box_a[1] + box_a[3]) / 2
    bx = (box_b[0] + box_b[2]) / 2
    by = (box_b[1] + box_b[3]) / 2
    return ((ax - bx) ** 2 + (ay - by) ** 2) ** 0.5

### 6-4. 메인 파이프라인

- 매 프레임 이중 배경 모델을 갱신하고(가벼운 연산이라 실시간에 무리 없음),
  **1초(`check_interval_sec`)마다** MobileNet-SSD로 사람 위치를 확인해서 체류시간을 누적합니다.
- 어떤 사람도 `persist_seconds`(기본 8초) 이상 머물지 않았다면 그 체크 타이밍엔 아무 것도
  하지 않고 넘어갑니다(=변동 없으면 저장 안 함).
- `persist_seconds` 이상 머문 사람이 있으면: 근처(`object_proximity_px` 이내)에 새로 생긴
  물체 후보가 있는지 확인 → 있으면 YOLO로 봉투/낱개쓰레기 분류 →
  **봉투면 문제 없음**, **낱개 쓰레기면 위반 확정 + 그 사람만 크롭해서 저장 + 로그 기록**.

In [ ]:
import csv
import time
from datetime import datetime


def run_realtime_dumping_pipeline(
    video_source,
    trash_model,
    person_detector,
    snapshot_dir="dumping_events",
    log_path="dumping_log.csv",
    check_interval_sec=1.0,    # 몇 초 간격으로 사람 체류시간을 점검할지
    persist_seconds=8.0,       # 사람이 이 시간 이상 같은 자리에 머물러야 판정 대상이 됨
    diff_min_area=800,         # 변화로 인정할 최소 픽셀 면적
    alpha_short=0.35,          # 단기 배경 적응 속도 (클수록 빨리 흡수)
    alpha_long=0.03,           # 장기 배경 적응 속도 (작을수록 오래 기억)
    diff_thresh=25,            # 배경과의 차이를 변화로 볼 픽셀 임계값
    ground_ratio=0.3,          # 화면 하단 몇 %까지를 "바닥"으로 볼지
    trash_conf=0.3,            # 물체 분류 최소 신뢰도
    person_iou_threshold=0.3,  # 사람 박스를 같은 사람으로 볼 IoU 기준
    object_proximity_px=180,   # 사람-물체 후보가 이 거리(px) 안이면 "그 사람이 놓은 물체"로 간주
    show_preview=False,        # 로컬(모니터 있는) 환경에서만 True 권장
    max_seconds=None,          # 테스트용: 지정 시간(초)만 돌리고 종료. 실시간 무한 실행은 None
):
    os.makedirs(snapshot_dir, exist_ok=True)

    cap = cv2.VideoCapture(int(video_source) if str(video_source).isdigit() else video_source)
    if not cap.isOpened():
        raise RuntimeError(f"영상을 열 수 없습니다: {video_source}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    check_interval_frames = max(1, int(round(fps * check_interval_sec)))

    dbs = DualBackgroundSubtractor(alpha_short=alpha_short, alpha_long=alpha_long, diff_thresh=diff_thresh)
    object_tracker = ChangeTracker(iou_threshold=0.3, max_missed=2)
    person_tracker = ChangeTracker(iou_threshold=person_iou_threshold, max_missed=2)

    frame_idx = 0
    log_rows = []
    start_time = time.time()

    print("=== 실시간 불법 투기 감지 시작 ===")
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame_idx += 1
        now_sec = frame_idx / fps

        gray = cv2.GaussianBlur(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (5, 5), 0)
        static_mask = dbs.apply(gray)

        object_boxes = []
        if static_mask is not None:
            for box in boxes_from_mask(static_mask, diff_min_area):
                if is_on_ground(box, frame.shape[0], ground_ratio):
                    object_boxes.append(box)

        # 배경 갱신 + 물체 후보 추적은 매 프레임 수행 (가벼운 연산)
        object_tracker.update(object_boxes, now_sec)

        # 무거운 모델 호출(사람 탐지/물체 분류)은 1초 간격으로만 확인
        if frame_idx % check_interval_frames == 0:
            person_boxes = person_detector(frame)
            person_tracker.update(person_boxes, now_sec)

            pending_people = [
                (pid, p) for pid, p in person_tracker.candidates.items()
                if not p["resolved"] and (p["last_seen"] - p["first_seen"]) >= persist_seconds
            ]
            if pending_people:
                active_objects = list(object_tracker.candidates.items())
                for pid, person in pending_people:
                    # 이 사람 근처에 있는(=이 사람이 내려놓았을 법한) 물체 후보를 찾는다
                    nearby = [
                        (oid, o) for oid, o in active_objects
                        if _box_distance(person["bbox"], o["bbox"]) <= object_proximity_px
                    ]
                    if not nearby:
                        continue  # 아직 근처에 물체가 없음 -> 다음 체크 타이밍에 재시도

                    # 가장 가까운 물체 하나를 이 사람이 놓은 물체로 간주
                    oid, obj = min(nearby, key=lambda kv: _box_distance(person["bbox"], kv[1]["bbox"]))
                    ox1, oy1, ox2, oy2 = obj["bbox"]
                    roi = frame[max(0, oy1):oy2, max(0, ox1):ox2]
                    label, score, is_bag = classify_dropped_object(trash_model, roi, trash_conf)

                    if label is None:
                        continue  # 아직 뭔지 판별 안 됨 -> 다음 체크 타이밍에 재시도

                    person["resolved"] = True
                    object_tracker.candidates[oid]["resolved"] = True

                    if is_bag:
                        print(f"[정상] person_id={pid} 봉투({label} {score:.2f}) 배출 - 문제 없음")
                        continue

                    # --- 위반 확정: 화면 전체가 아니라 그 사람만 크롭해서 저장 ---
                    person_crop = crop_person(frame, person["bbox"])
                    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                    fname = f"person_{pid}_{frame_idx}.jpg"
                    cv2.imwrite(os.path.join(snapshot_dir, fname), person_crop)
                    px1, py1, px2, py2 = person["bbox"]
                    log_rows.append([timestamp, f"{now_sec:.1f}", pid, label, f"{score:.2f}",
                                      px1, py1, px2, py2, fname])
                    print(f"[투기 확정] person_id={pid} 물체={label}({score:.2f}) "
                          f"체류={person['last_seen']-person['first_seen']:.1f}초, "
                          f"사람 사진 저장={fname}")

        if show_preview:
            cv2.imshow("realtime-dumping-detector", frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
        if max_seconds is not None and (time.time() - start_time) > max_seconds:
            break

    cap.release()
    if show_preview:
        cv2.destroyAllWindows()

    with open(log_path, "w", newline="", encoding="utf-8-sig") as f:
        writer_csv = csv.writer(f)
        writer_csv.writerow(["timestamp", "video_sec", "person_id", "object_class", "confidence",
                              "x1", "y1", "x2", "y2", "person_snapshot_file"])
        writer_csv.writerows(log_rows)

    print(f"=== 종료: 총 {len(log_rows)}건의 투기 확정 이벤트 ===")
    print("로그:", log_path, "| 저장 폴더:", snapshot_dir)
    return log_rows

## 7. 실행

- **웹캠 실시간**: `video_source=0`, `max_seconds=None` (직접 멈출 때까지 계속 실행)
- **저장된 영상 파일**: `video_source="파일경로.mp4"`
- `show_preview=True`는 로컬(모니터가 있는) 환경에서만 사용하세요. Colab이나 원격
  노트북에서는 `False`로 두고, 결과는 `dumping_events/` 폴더에 저장된 **위반자 사진**과
  `dumping_log.csv`로 확인합니다.

In [ ]:
VIDEO_SOURCE = 0  # 웹캠 실시간 감지. 영상 파일을 쓰려면 "파일경로.mp4"로 교체

person_detector = make_mobilenet_ssd_detector(ssd_net, conf_threshold=0.5)

run_realtime_dumping_pipeline(
    video_source=VIDEO_SOURCE,
    trash_model=trained_model,
    person_detector=person_detector,
    snapshot_dir="dumping_events",
    log_path="dumping_log.csv",
    check_interval_sec=1.0,
    persist_seconds=8.0,
    ground_ratio=0.3,
    show_preview=False,
    max_seconds=None,  # 실시간으로 계속 돌리려면 None, 몇 분만 테스트하려면 초 단위 숫자
)

## 8. 테스트

실제 학습 없이도 핵심 로직을 검증할 수 있는 테스트 3가지 + 실제 모델로 돌리는 스모크 테스트
1가지, 총 4가지를 준비했습니다.

1. **봉투 배출 테스트**: 사람이 8초 이상 머문 뒤 근처에 "봉투"류 물체가 생김 → **정상**으로
   판정되어 확정 이벤트가 0건이어야 함
2. **낱개 쓰레기 투기 테스트**: 사람이 8초 이상 머문 뒤 근처에 "낱개 쓰레기"류 물체가 생김 →
   **위반 확정** 1건 + 그 사람 크롭 사진 파일이 실제로 저장되는지 확인
3. **계속 이동하는 사람 테스트**: 위치가 계속 바뀌는(정지하지 않는) 사람은 체류시간이
   8초까지 쌓이지 않아야 함 → 확정 이벤트 0건
4. **샘플 영상 스모크 테스트**: 실제 학습된 `trained_model`과 실제 `ssd_net`으로 공개
   샘플 영상을 돌려서 전체 파이프라인이 에러 없이 끝까지 도는지 확인 (사람만 지나다니고
   8초 이상 머물지 않는 영상이라 이벤트 0건이 정상입니다)

In [ ]:
# --- 로직 테스트용 헬퍼 (더미 모델 + 합성 영상/사람 위치) ---
import numpy as np


def _make_synthetic_frames(n=30, obj_appear=10, obj_box=(150, 180, 200, 220)):
    '''화면 속 물체(봉투/쓰레기)가 obj_appear 시점부터 같은 자리에 나타나서 안 없어짐.'''
    H, W = 240, 320
    rng = np.random.default_rng(0)
    frames = []
    for i in range(n):
        base = np.full((H, W, 3), 40, dtype=np.uint8)
        noise = rng.integers(-3, 3, base.shape, endpoint=True).astype(np.int16)
        frame = np.clip(base.astype(np.int16) + noise, 0, 255).astype(np.uint8)
        if i >= obj_appear:
            x1, y1, x2, y2 = obj_box
            cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 255, 255), -1)
        frames.append(frame)
    return frames


class _FakeCapture:
    def __init__(self, frames, fps=1.0):
        self.frames = frames
        self.idx = 0
        self.fps = fps

    def isOpened(self):
        return True

    def get(self, prop):
        return self.fps  # fps=1 -> 프레임 하나가 곧 1초 샘플

    def read(self):
        if self.idx >= len(self.frames):
            return False, None
        f = self.frames[self.idx]
        self.idx += 1
        return True, f

    def release(self):
        pass


class _DummyBoxes:
    def __init__(self, cls, conf):
        self.cls = np.array(cls)
        self.conf = np.array(conf)

    def __len__(self):
        return len(self.cls)


def _make_dummy_trash_model(label):
    '''실제 학습된 모델 없이 로직만 검증하기 위한 가짜 모델 (항상 지정한 라벨로 인식)'''
    class _M:
        names = {0: label, 1: "person"}

        def predict(self, img, verbose=False, conf=0.3):
            class _R:
                pass
            r = _R()
            r.boxes = _DummyBoxes(cls=[0], conf=[0.9])
            return [r]
    return _M()


def _stationary_person_detector(box=(140, 170, 210, 230), start_at=4):
    '''start_at 시점부터 항상 같은 자리에 사람이 있는 것처럼 반환 (정지한 사람 시뮬레이션)'''
    state = {"i": 0}

    def _detect(frame):
        i = state["i"]
        state["i"] += 1
        return [box] if i >= start_at else []
    return _detect


def _walking_person_detector():
    '''매 체크마다 크게 위치가 점프하는 사람 (계속 걸어다니는 사람 시뮬레이션 -> 같은 ID로 안 이어짐)'''
    state = {"i": 0}

    def _detect(frame):
        i = state["i"]
        state["i"] += 1
        x = 10 + (i * 70) % 260
        return [(x, 170, x + 40, 230)]
    return _detect

In [ ]:
# --- 테스트 1: 사람이 8초+ 머문 뒤 근처에 "봉투"가 생김 -> 문제 없음(0건) ---
_orig_video_capture = cv2.VideoCapture
frames = _make_synthetic_frames()
cv2.VideoCapture = lambda *a, **k: _FakeCapture(frames, fps=1.0)
try:
    rows_1 = run_realtime_dumping_pipeline(
        video_source=0, trash_model=_make_dummy_trash_model("garbage_bag"),
        person_detector=_stationary_person_detector(),
        snapshot_dir="test_events_1", log_path="test_log_1.csv",
        check_interval_sec=1.0, persist_seconds=8.0, diff_min_area=400, object_proximity_px=200,
    )
finally:
    cv2.VideoCapture = _orig_video_capture

assert len(rows_1) == 0, f"봉투 배출은 0건이어야 합니다 (실제: {len(rows_1)}건)"
print("테스트 1 통과: 봉투 배출은 확정 이벤트 0건 ✅")

In [ ]:
# --- 테스트 2: 사람이 8초+ 머문 뒤 근처에 "낱개 쓰레기"가 생김 -> 위반 확정 + 사람 사진 저장 ---
frames = _make_synthetic_frames()
cv2.VideoCapture = lambda *a, **k: _FakeCapture(frames, fps=1.0)
try:
    rows_2 = run_realtime_dumping_pipeline(
        video_source=0, trash_model=_make_dummy_trash_model("garbage"),
        person_detector=_stationary_person_detector(),
        snapshot_dir="test_events_2", log_path="test_log_2.csv",
        check_interval_sec=1.0, persist_seconds=8.0, diff_min_area=400, object_proximity_px=200,
    )
finally:
    cv2.VideoCapture = _orig_video_capture

assert len(rows_2) == 1, f"1건의 투기 확정 이벤트가 발생해야 합니다 (실제: {len(rows_2)}건)"
saved_path = os.path.join("test_events_2", rows_2[0][-1])
assert os.path.exists(saved_path), "위반자 크롭 사진 파일이 저장되어야 합니다"
print(f"테스트 2 통과: 낱개 쓰레기 투기 1건 확정, 사람 사진 저장됨 ({saved_path}) ✅")

In [ ]:
# --- 테스트 3: 계속 이동하는(정지하지 않는) 사람 -> 체류시간이 8초까지 안 쌓여서 0건 ---
frames = _make_synthetic_frames()
cv2.VideoCapture = lambda *a, **k: _FakeCapture(frames, fps=1.0)
try:
    rows_3 = run_realtime_dumping_pipeline(
        video_source=0, trash_model=_make_dummy_trash_model("garbage"),
        person_detector=_walking_person_detector(),
        snapshot_dir="test_events_3", log_path="test_log_3.csv",
        check_interval_sec=1.0, persist_seconds=8.0, diff_min_area=400, object_proximity_px=200,
    )
finally:
    cv2.VideoCapture = _orig_video_capture

assert len(rows_3) == 0, f"계속 움직이면 0건이어야 합니다 (실제: {len(rows_3)}건)"
print("테스트 3 통과: 계속 이동하는 사람은 체류시간이 안 쌓여 확정 이벤트 0건 ✅")

In [ ]:
# --- 테스트 4: 실제 학습 모델 + 실제 MobileNet-SSD + 샘플 영상으로 스모크 테스트 ---
import urllib.request

if not os.path.exists("test_video.avi"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/vtest.avi",
        "test_video.avi",
    )

real_person_detector = make_mobilenet_ssd_detector(ssd_net, conf_threshold=0.5)

test_rows = run_realtime_dumping_pipeline(
    video_source="test_video.avi",
    trash_model=trained_model,
    person_detector=real_person_detector,
    snapshot_dir="test_events_sample",
    log_path="test_log_sample.csv",
    check_interval_sec=1.0,
    persist_seconds=8.0,
    max_seconds=60,  # 테스트니까 60초만
)
print("테스트 4 완료 (에러 없이 끝까지 실행됨). 감지된 이벤트 수:", len(test_rows))
print("(참고: 이 샘플 영상엔 사람이 지나다니기만 하고 8초 이상 머물지 않아서 0건이 정상입니다)")